# VisClick — Phase 4.4 / D-03: UDA — simplified Cross-Domain Adaptive Teacher

**Goal.** Adapt the source-trained YOLOv8s detector to the desktop domain using **unlabelled** desktop images only. Method: simplified offline Adaptive Teacher (after Li et al., 2022).

**Corpus.** The unlabelled target is built from data we already hold:

- **ScreenSpot** (Cheng et al., 2024) desktop slice — ~334 macOS + Windows screenshots, downloaded from HuggingFace `rootsautomation/ScreenSpot`. We use the *images only*; the instructions and GT boxes are ignored at adaptation time (UDA is label-free).
- **`samples/desktop_seed/`** — 50 hand-selected desktop screenshots in the repo.

Total ~384 unlabelled desktop images. ScreenSpot already powers the D-07 CPV evaluation, so the cache is reused. Note this is a *test-time* benchmark; reusing it as an adaptation corpus is unconventional but defensible because UDA only consumes the images, never the labels. The protocol caveat is named in the report.

**Why simplified.** The full Adaptive Teacher of Li et al. 2022 runs an online EMA teacher-student loop inside the detector's training stage, with mixed batches of weakly-augmented teacher and strongly-augmented student forwards. Implementing that inside Ultralytics' YOLO trainer requires hooking the optimizer and reordering the dataloader, which is brittle. The simplified version below preserves the **structural idea** of teacher-student mutual learning while running offline pseudo-labelling between standard YOLO training runs.

**Pipeline (3 outer iterations of teacher → pseudo-labels → student):**
1. Mount Drive → `git pull` → install.
2. Build the unlabelled target corpus from ScreenSpot + seed.
3. Iteration `t`:
   - Teacher `T_t` generates pseudo-labels on the unlabelled corpus at confidence ≥ 0.30 (filter).
   - Student `S_t` = retrain YOLOv8s for 10 epochs on (source GT pool + pseudo-labelled target).
   - Update: `T_{t+1}` = `S_t`.
4. Final evaluation: CPV on ScreenSpot, mAP on hand-corrected.
5. Write `reports/tables/uda_adaptive_teacher.csv` (one row per iteration + final).
6. Publish.

**Compute reality.** Each outer iteration: ~5 min for pseudo-labelling + ~20 min training = ~25 min. Three iterations = ~75 min. Fits one Colab Free session with margin.

**Honest narrative for the report.** The simplification is named explicitly in Section 6 and Section 8: results from this offline variant are a lower bound on what the full online Adaptive Teacher would achieve.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)
print("REPORT git_head =", subprocess.check_output(
    ["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
import sys, subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics", "pillow", "opencv-python", "matplotlib", "pi-heif"],
    check=False,
)
import torch, ultralytics
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("ultralytics:", ultralytics.__version__)


## 13.1 — Bootstrap source weights and the unlabelled desktop target corpus

The teacher starts as `best_source_v8s.pt`. The unlabelled *target* corpus is built from two sources we already hold:

- **ScreenSpot desktop slice** (Cheng et al., 2024) downloaded via HuggingFace `rootsautomation/ScreenSpot`. We extract the images to PNGs and ignore the instructions / GT boxes (UDA is label-free at adaptation time).
- **`samples/desktop_seed/`** in the repo (50 images).

Together: ~384 unlabelled desktop screenshots. The hand-corrected set is unzipped for evaluation only.


In [ ]:
import os, glob, shutil, zipfile, tempfile

DRIVE        = "/content/drive/MyDrive/visclick"
SOURCE_WTS   = os.path.join(DRIVE, "weights", "baseline_source", "best_source_v8s.pt")
UDA_DIR      = os.path.join(DRIVE, "weights", "uda_at")
REPORTS_TBL  = os.path.join(DRIVE, "reports", "tables")
os.makedirs(UDA_DIR, exist_ok=True)
os.makedirs(REPORTS_TBL, exist_ok=True)

assert os.path.isfile(SOURCE_WTS), f"Source weights missing: {SOURCE_WTS}"

# --- Materialise ScreenSpot desktop slice as PNGs on local disk. ---
SCREENSPOT_DIR = "/content/screenspot_desktop_pngs"
os.makedirs(SCREENSPOT_DIR, exist_ok=True)

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=False)
from datasets import load_dataset

HF_CACHE = os.path.join(tempfile.gettempdir(), "visclick_hf_cache")
ds = load_dataset("rootsautomation/ScreenSpot", split="test", cache_dir=HF_CACHE)
print(f"loaded ScreenSpot rows = {len(ds)}")

n_written = 0
for i, row in enumerate(ds):
    if row.get("data_source") not in ("macos", "windows"):
        continue
    out = os.path.join(SCREENSPOT_DIR, f"ss_{i:04d}.png")
    if os.path.isfile(out):
        continue
    row["image"].save(out)
    n_written += 1
print(f"REPORT screenspot_pngs | written = {n_written} | dir = {SCREENSPOT_DIR}")

# --- Collect unlabelled target paths. ---
TGT_DIRS = [
    SCREENSPOT_DIR,
    "/content/visclick/samples/desktop_seed",
]
TARGET_PATHS = []
for d in TGT_DIRS:
    if not os.path.isdir(d):
        continue
    for f in os.listdir(d):
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            TARGET_PATHS.append(os.path.join(d, f))
TARGET_PATHS = sorted(set(TARGET_PATHS))
print(f"REPORT target_corpus | n = {len(TARGET_PATHS)} | first = {TARGET_PATHS[:2]}")
assert len(TARGET_PATHS) >= 100, f"too few unlabelled targets ({len(TARGET_PATHS)})."

# Hand-corrected set (eval only).
HC_ZIP  = "/content/visclick/datasets/handcorrected_desktop_test/visclick3.yolov8.zip"
HC_ROOT = "/content/hc"
if not os.path.isdir(os.path.join(HC_ROOT, "train", "images")):
    with zipfile.ZipFile(HC_ZIP, "r") as zf:
        zf.extractall(HC_ROOT)

CLASSES = ["button", "text", "text_input", "icon", "menu", "checkbox"]


## 13.2 — Three outer iterations of pseudo-label → train → swap

For each outer iteration `t`:
1. Run teacher `T_t` on all unlabelled target images → write YOLO `.txt` pseudo-labels with confidence ≥ 0.30.
2. Build a mixed dataset: source GT pool (already on Drive) + pseudo-labelled target.
3. Train YOLOv8s for 10 epochs starting from `T_t`.
4. The trained model becomes `T_{t+1}`.

The `freeze=0` argument means we fine-tune the whole network, which is what Adaptive Teacher does in its student branch.


In [ ]:
import time, yaml, tarfile
from ultralytics import YOLO

# Source GT pool: the 6-class source_train set assembled by 04_assemble_source.ipynb.
# Fast-restore it from the tiny Drive bundles + manifests, same pattern as 05_train_source.ipynb.
UNIFIED      = os.path.join(DRIVE, "data", "unified")
BUNDLES      = os.path.join(DRIVE, "data", "source_train_bundles")
SOURCE_DATA  = "/content/source_train"        # local, ephemeral
SRC_YAML     = os.path.join(SOURCE_DATA, "data.yaml")
SPLITS       = ["train", "val"]


def _bootstrap_source_train():
    if os.path.isfile(SRC_YAML):
        return
    os.makedirs(SOURCE_DATA, exist_ok=True)
    for sp in SPLITS:
        b = os.path.join(BUNDLES, f"{sp}.tar.gz")
        assert os.path.isfile(b), f"Drive bundle missing: {b} (run 04_assemble_source.ipynb first)"
        with tarfile.open(b, "r:gz") as tf:
            tf.extractall(SOURCE_DATA)
        manifest = os.path.join(SOURCE_DATA, "manifests", f"{sp}.txt")
        assert os.path.isfile(manifest), f"manifest missing in bundle: {manifest}"
        with open(manifest) as fh:
            names = [ln.strip() for ln in fh if ln.strip()]
        src_img_dir = os.path.join(UNIFIED, sp, "images")
        dst_img_dir = os.path.join(SOURCE_DATA, "images", sp)
        os.makedirs(dst_img_dir, exist_ok=True)
        for fn in names:
            dst = os.path.join(dst_img_dir, fn)
            if os.path.exists(dst):
                continue
            try:
                os.symlink(os.path.join(src_img_dir, fn), dst)
            except OSError:
                pass
    with open(SRC_YAML, "w") as fh:
        yaml.safe_dump({"path": SOURCE_DATA, "train": "images/train", "val": "images/val",
                        "nc": len(CLASSES), "names": CLASSES}, fh, sort_keys=False)
    print(f"bootstrap done at {SOURCE_DATA}")


_bootstrap_source_train()
assert os.path.isdir(os.path.join(SOURCE_DATA, "images", "train")), "source_train bootstrap failed"

PSEUDO_CONF = 0.30
N_OUTER = 3
EPOCHS_PER_ITER = 10
IMGSZ = 640


def write_pseudo_labels(model, work_dir):
    img_dir = os.path.join(work_dir, "images", "train")
    lbl_dir = os.path.join(work_dir, "labels", "train")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    n_with_box = 0
    for path in TARGET_PATHS:
        results = model.predict(path, imgsz=IMGSZ, conf=PSEUDO_CONF, verbose=False)
        r = results[0]
        if len(r.boxes) == 0:
            continue
        stem = os.path.splitext(os.path.basename(path))[0]
        out_img = os.path.join(img_dir, os.path.basename(path))
        out_lbl = os.path.join(lbl_dir, stem + ".txt")
        if not os.path.isfile(out_img):
            shutil.copy2(path, out_img)
        H, W = r.orig_shape
        with open(out_lbl, "w") as fh:
            for cls, xyxy in zip(r.boxes.cls.tolist(), r.boxes.xyxyn.tolist()):
                x1, y1, x2, y2 = xyxy
                cx = (x1 + x2) / 2; cy = (y1 + y2) / 2
                bw = x2 - x1; bh = y2 - y1
                fh.write(f"{int(cls)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
        n_with_box += 1
    return n_with_box


def build_mixed_yaml(target_dir, out_yaml):
    # Source uses its own YAML; we point YOLOv8 at the source for both train and val,
    # then UNIQUELY train on the union by listing two image roots.
    # Ultralytics supports multi-path train via a list in the yaml.
    data = {
        "path": "/",
        "train": [
            os.path.join(SOURCE_DATA, "images", "train"),
            os.path.join(target_dir, "images", "train"),
        ],
        "val": os.path.join(SOURCE_DATA, "images", "val"),
        "names": CLASSES,
        "nc": len(CLASSES),
    }
    with open(out_yaml, "w") as fh:
        yaml.safe_dump(data, fh)


HISTORY = []
teacher_weights = SOURCE_WTS

for t in range(1, N_OUTER + 1):
    print(f"\n=== Outer iteration {t}/{N_OUTER} ===")
    work_dir = f"/content/uda_at_iter{t}"
    teacher = YOLO(teacher_weights)
    n_box = write_pseudo_labels(teacher, work_dir)
    print(f"  pseudo-labels: {n_box} / {len(TARGET_PATHS)} images had >= 1 box at conf {PSEUDO_CONF}")

    data_yaml = os.path.join(work_dir, "data.yaml")
    build_mixed_yaml(work_dir, data_yaml)

    t0 = time.time()
    student = YOLO(teacher_weights)
    student.train(
        data=data_yaml,
        epochs=EPOCHS_PER_ITER,
        imgsz=IMGSZ,
        batch=8,
        project=UDA_DIR,
        name=f"iter{t}",
        verbose=False,
        plots=False,
    )
    elapsed = time.time() - t0
    new_weights = os.path.join(UDA_DIR, f"iter{t}", "weights", "best.pt")
    if not os.path.isfile(new_weights):
        new_weights = os.path.join(UDA_DIR, f"iter{t}", "weights", "last.pt")
    HISTORY.append({"iter": t, "n_pseudo_imgs": n_box, "weights": new_weights, "elapsed_s": elapsed})
    print(f"REPORT uda_at_iter | t = {t} | n_pseudo = {n_box} | elapsed_s = {elapsed:.1f}")

    teacher_weights = new_weights


## 13.3 — Final evaluation: CPV on ScreenSpot + hand-corrected mAP

The final teacher (after `N_OUTER` rounds) is evaluated against the same protocol as every other adapter: ScreenSpot CPV for instruction-grounded success rate, hand-corrected for per-element recall.


In [ ]:
import subprocess, tempfile, csv

final_weights = HISTORY[-1]["weights"]
print("REPORT uda_at_final | weights =", final_weights)

# Export ONNX for the eval scripts.
onnx_out = final_weights.replace(".pt", ".onnx")
if not os.path.isfile(onnx_out):
    YOLO(final_weights).export(format="onnx", imgsz=IMGSZ, dynamic=False, opset=12)

with tempfile.TemporaryDirectory() as tmp:
    ss_csv = os.path.join(tmp, "ss.csv")
    subprocess.run([
        sys.executable, "/content/visclick/scripts/run_cpv_screenspot.py",
        "--weights", onnx_out, "--out", ss_csv,
    ], check=True)
    with open(ss_csv) as fh:
        head = next(fh); ss_overall = next(fh).strip().split(",")
    CPV_SS = float(ss_overall[-1])

    hc_csv = os.path.join(tmp, "hc.csv")
    subprocess.run([
        sys.executable, "/content/visclick/scripts/run_cpv.py",
        "--weights", onnx_out, "--out", hc_csv,
    ], check=True)
    with open(hc_csv) as fh:
        next(fh)
        hc_overall = None
        for line in fh:
            parts = line.strip().split(",")
            if parts and parts[0] == "OVERALL":
                hc_overall = parts; break
    CPV_HC = float(hc_overall[-1]) if hc_overall else float("nan")

print(f"REPORT uda_at_eval | cpv_screenspot = {CPV_SS:.2f} | cpv_handcorrected = {CPV_HC:.2f}")


## 13.4 — Write `uda_adaptive_teacher.csv`

In [ ]:
OUT_CSV = "/content/visclick/reports/tables/uda_adaptive_teacher.csv"
with open(OUT_CSV, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["iter", "n_pseudo_imgs", "elapsed_s",
                "cpv_screenspot_%", "cpv_handcorrected_%"])
    for h in HISTORY[:-1]:
        w.writerow([h["iter"], h["n_pseudo_imgs"], f"{h['elapsed_s']:.1f}", "", ""])
    h = HISTORY[-1]
    w.writerow([h["iter"], h["n_pseudo_imgs"], f"{h['elapsed_s']:.1f}",
                f"{CPV_SS:.2f}", f"{CPV_HC:.2f}"])
print(f"REPORT step = WRITE_CSV | path = {OUT_CSV}")
shutil.copy2(OUT_CSV, os.path.join(REPORTS_TBL, "uda_adaptive_teacher.csv"))


## 13.5 — Publish to git

**Before running the cell below**, paste your GitHub personal-access token (PAT) into the `TOKEN = "PASTE_GITHUB_TOKEN_HERE"` line. The token lives only in Colab runtime memory and disappears when the runtime is recycled.


In [ ]:
# ---------------------------------------------------------------------------
# Paste your GitHub personal-access token (PAT) on the line below before
# running this cell. Replace the placeholder string. The token is NOT
# committed to git or saved anywhere; it lives only in the Colab runtime
# memory for this session, and disappears when the runtime is recycled.
# ---------------------------------------------------------------------------
TOKEN = "PASTE_GITHUB_TOKEN_HERE"

import os, subprocess

REPO_ROOT = "/content/visclick"
ARTIFACTS = [
    'reports/tables/uda_adaptive_teacher.csv',
]

assert TOKEN and TOKEN != "PASTE_GITHUB_TOKEN_HERE", (
    "Paste your GitHub personal-access token into the TOKEN variable above "
    "before running this cell."
)

for rel in ARTIFACTS:
    p = os.path.join(REPO_ROOT, rel)
    assert os.path.exists(p), f"Missing artifact in repo clone: {p}. Run the previous section first."
    print(f"OK  {p}  ({os.path.getsize(p)} bytes)")


def run(cmd, **kw):
    r = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print("STDOUT:", r.stdout)
        print("STDERR:", r.stderr)
        raise RuntimeError(f"git command failed: {' '.join(cmd)}")
    return r.stdout


run(["git", "config", "user.email", "hiran@iit.ac.lk"])
run(["git", "config", "user.name",  "Hiran Abeywardhana"])

run(["git", "add", *ARTIFACTS])

status = run(["git", "status", "--porcelain"])
if not status.strip():
    print("REPORT step = GIT_PUBLISH | status = NOTHING_TO_COMMIT")
else:
    run(["git", "commit", "-m", 'D-03: simplified Adaptive Teacher UDA results'])
    url = f"https://{TOKEN}@github.com/HiranMadhu/visclick.git"
    push = subprocess.run(["git", "push", url, "HEAD:main"],
                          cwd=REPO_ROOT, capture_output=True, text=True)
    if push.returncode != 0:
        print("PUSH STDERR:", push.stderr)
        raise RuntimeError("git push failed")
    print("REPORT step = GIT_PUBLISH | status = PUSHED")
